# TRELLIS.2 shape stage on Colab

**Licence finding:** Microsoft's code and 4B weights are MIT, but the trained
image conditioner requires **Meta DINOv3's custom licence and gated access**.
This is not a permissive-only pipeline. Read the
[DINOv3 licence](https://github.com/facebookresearch/dinov3/blob/main/LICENSE.md)
and request access to [the encoder](https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m)
before using this notebook. BRIA RMBG-2.0 and NVIDIA's noncommercial renderers
are excluded. Background removal uses rembg / Apache-2.0 U²-Net instead.

On **Colab Pro**, use **Runtime → Change runtime type → Python 3 → GPU → A100**
(recommended, 40 GB class) or **L4 (24 GB)**. Availability and compute units vary;
Pro does not guarantee either GPU. Upstream requires at least 24 GB and tests
A100/H100; L4 is an unverified minimum-memory target. Default: 512 and CPU offload.
For A100 you may select `1024_cascade`. **T4 (16 GB) is unsupported**: switch
to L4/A100 or wait for availability. No speculative T4 workaround is offered.

Edit the parameter cell, then **Runtime → Run all**. Installation can take
tens of minutes and downloads several GB of weights. A separate Python
environment avoids kernel restarts. Enter a read-only Hugging Face token
when asked; it must belong to an account approved for the encoder.
Upload `front.png` and optionally `back.png`, `left.png`, `right.png` together.
Full-body, standing, consistent A-pose views are best. Only **front** conditions
the shape: upstream has no supported multi-image API. Other views get cutouts
and orthographic camera fits for the subsequent texture stage.

Output: `trellis2_shape.zip` containing `mesh.glb`, `meta.json`, cutouts and
masks. Save/unzip it into your gitignored `user-data/twin/out/shape/` locally.
Geometry is untextured/unrigged, in metres, +Y up, facing +Z, feet at y=0,
with total mesh height (including soles/hair) equal to `HEIGHT_CM`.
This is a shape intermediate; the rig stage must create the final `twin.glb`
and its `asset.extras.dtTwin` contract. Inspect orientation locally and rerun
with source-axis overrides if the automatic anatomical heuristic is wrong.

**Privacy:** images are processed on Google's VM for this session only.
This is the user-operated Colab exception to twin-lab's local-only workflow.
No Drive is mounted and no image/mesh previews are displayed or embedded in
notebook outputs. Downloads stay on your device. Session files are deleted
in `finally` after the archive reaches browser memory, including on ordinary
errors/cancellation. This removes VM files; it is not secure erasure or a
guarantee about Google's retention. If the kernel/VM crashes or is forcibly
stopped, use **Runtime → Disconnect and delete runtime**. Do that after every
session, and clear notebook outputs before sharing it.


In [ ]:
# Parameters: edit these before Runtime > Run all.
HEIGHT_CM = 178.0
SEED = 42
PIPELINE_TYPE = "512"  # "512" for L4; "1024_cascade" optionally on A100
STEPS = 12
TARGET_FACES = 80000
MAX_NUM_TOKENS = 49152
SOURCE_UP = "auto"  # "auto" or one of "+X", "-X", "+Y", "-Y", "+Z", "-Z"
SOURCE_FORWARD = "auto"  # Override BOTH axes if detection is wrong.
FLIP_FORWARD = False  # Rotate 180 degrees about final +Y when front is reversed.
# Strict permissive-only use cannot run this model's trained image encoder.
# Set True only after independently reviewing/accepting Meta's DINOv3 terms.
DINO_LICENSE_ACKNOWLEDGED = False


## Install and check access before uploading images

The licence prerequisite deliberately fails before installation or upload
until `DINO_LICENSE_ACKNOWLEDGED` is set. There is no automatic acceptance of
third-party terms. The worker uses the pinned shape sampling/decoding substeps,
rather than `pipeline.run`, which also loads BRIA and generates PBR attributes.
O-Voxel's package entry point is restricted to geometry conversion in this VM
to avoid an eager renderer import. GPU installation/inference remain untested
by the notebook authors; compiler failures stop immediately.


In [ ]:
"""Colab setup and privacy helpers, embedded in the portable notebook."""

import gc
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

TRELLIS_REVISION = "75fbf0183001ed9876c8dbb35de6b68552ee08bd"
CUMESH_REVISION = "12289e1062f0603f2f0d0771b02e1395d247f26f"
FLEXGEMM_REVISION = "6dd94a859c26ee8246888502eada3dd8ad85532e"
SPARSE_DECODER_REVISION = "25e0d31ffbebe4b5a97464dd851910efc3002d96"
DINO_REVISION = "ea8dc2863c51be0a264bab82070e3e8836b02d51"
BUILD_ROOT = Path("/content/dt-trellis2-software")
ENV_PYTHON = BUILD_ROOT / "venv/bin/python"
TRELLIS_SOURCE = BUILD_ROOT / "TRELLIS.2"


def run_command(arguments, **kwargs):
    # No shell interpolation, and every compiler / installer failure stops the cell.
    return subprocess.run([str(item) for item in arguments], check=True, **kwargs)


def pip_install(*arguments):
    run_command([ENV_PYTHON, "-m", "pip", "install", *arguments])


def clone_pinned(url, revision, destination):
    if not destination.exists():
        run_command(["git", "clone", "--filter=blob:none", url, destination])
    run_command(["git", "-C", destination, "checkout", "--detach", revision])
    run_command(["git", "-C", destination, "submodule", "update", "--init", "--recursive"])


def install_software():
    if not (3, 10) <= sys.version_info[:2] <= (3, 12):
        raise RuntimeError("This pinned stack targets Python 3.10-3.12. Select a Colab runtime with Python 3.12.")
    memory = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], text=True
    ).strip()
    name, memory_mib = memory.splitlines()[0].rsplit(",", 1)
    print(f"GPU: {name.strip()}, {int(memory_mib)} MiB")
    if int(memory_mib) < 22000:
        raise RuntimeError("TRELLIS.2 requires a 24 GB class GPU. T4/16 GB is unsupported; select L4 or A100, then Run all again.")
    BUILD_ROOT.mkdir(parents=True, exist_ok=True)
    toolkit = Path("/usr/local/cuda-12.4")
    if not (toolkit / "bin/nvcc").exists():
        import urllib.request

        # Google Colab uses Ubuntu; install the exact upstream-recommended toolkit.
        release = Path("/etc/os-release").read_text()
        distro = "ubuntu2404" if 'VERSION_ID="24.04"' in release else "ubuntu2204"
        keyring = BUILD_ROOT / "cuda-keyring.deb"
        urllib.request.urlretrieve(
            f"https://developer.download.nvidia.com/compute/cuda/repos/{distro}/x86_64/cuda-keyring_1.1-1_all.deb",
            keyring,
        )
        run_command(["dpkg", "-i", keyring])
        run_command(["apt-get", "update", "-qq"])
        run_command(["apt-get", "install", "-y", "cuda-toolkit-12-4", "build-essential", "git", "libjpeg-dev"])
    os.environ.update({
        "CUDA_HOME": str(toolkit), "PATH": f"{toolkit}/bin:" + os.environ["PATH"],
        "MAX_JOBS": "2", "HF_HUB_DISABLE_TELEMETRY": "1", "DO_NOT_TRACK": "1",
        "ATTN_BACKEND": "flash_attn", "SPARSE_ATTN_BACKEND": "flash_attn",
        "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
        "TORCH_CUDA_ARCH_LIST": "8.9" if "L4" in name else "8.0",
    })
    if not ENV_PYTHON.exists():
        run_command([sys.executable, "-m", "venv", BUILD_ROOT / "venv"])
    # Isolated subprocess environment avoids replacing Colab kernel packages or restarting it.
    pip_install("--upgrade", "pip", "setuptools", "wheel")
    pip_install("torch==2.6.0", "torchvision==0.21.0", "--index-url", "https://download.pytorch.org/whl/cu124")
    pip_install(
        "numpy<3", "scipy", "pillow", "opencv-python-headless", "ninja", "packaging", "psutil",
        "trimesh>=4.6,<5", "transformers==4.57.3", "easydict", "tqdm", "einops", "safetensors",
        "huggingface_hub<1", "kornia", "timm", "networkx", "fast-simplification", "zstandard", "rembg[cpu]",
    )
    clone_pinned("https://github.com/microsoft/TRELLIS.2.git", TRELLIS_REVISION, TRELLIS_SOURCE)
    clone_pinned("https://github.com/JeffreyXiang/CuMesh.git", CUMESH_REVISION, BUILD_ROOT / "CuMesh")
    clone_pinned("https://github.com/JeffreyXiang/FlexGEMM.git", FLEXGEMM_REVISION, BUILD_ROOT / "FlexGEMM")
    # O-Voxel eagerly imports postprocess, which imports the noncommercial nvdiffrast.
    # Change only its package entry point in this VM; geometry conversion is unaffected.
    (TRELLIS_SOURCE / "o-voxel/o_voxel/__init__.py").write_text(
        '"""Shape-only entry point modified by Dijital Ikiz Colab notebook."""\nfrom . import convert\n', encoding="utf-8"
    )
    pip_install("flash-attn==2.7.3", "--no-build-isolation")
    for source in (BUILD_ROOT / "CuMesh", BUILD_ROOT / "FlexGEMM", TRELLIS_SOURCE / "o-voxel"):
        pip_install(str(source), "--no-build-isolation")
    run_command([ENV_PYTHON, "-c", "import torch, flash_attn, cumesh, flex_gemm; assert torch.cuda.is_available()"])


def download_and_wait(path):
    """Await the files.download browser transfer before deleting its source file.

    files.download itself returns before transfer. Capture its JavaScript promise;
    fail explicitly if Colab changes that API implementation.
    """
    import IPython.display
    from google.colab import files, output
    from unittest.mock import patch

    original_display = IPython.display.display
    instrumented = False

    def display_with_completion(*objects, **kwargs):
        nonlocal instrumented
        changed = []
        for item in objects:
            if isinstance(item, IPython.display.Javascript) and item.data.strip().startswith("download("):
                item = IPython.display.Javascript("window.dtShapeDownload = " + item.data)
                instrumented = True
            changed.append(item)
        return original_display(*changed, **kwargs)

    with patch.object(IPython.display, "display", display_with_completion):
        files.download(str(path))
    if not instrumented:
        raise RuntimeError("Colab download implementation changed; automatic transfer confirmation needs updating.")
    output.eval_js("window.dtShapeDownload.then(() => true)", timeout_sec=600)


def cleanup_session(path):
    if path is None:
        return
    resolved = Path(path).resolve()
    if resolved.parent != Path("/content") or not resolved.name.startswith("dt-shape-session-"):
        raise ValueError("Refusing cleanup outside the notebook's private session directory")
    if resolved.exists():
        shutil.rmtree(resolved)
    gc.collect()
    print("Privacy cleanup complete: uploaded images, cutouts, meshes, metadata and archive removed from the VM.")


In [ ]:
import math
from getpass import getpass

if not DINO_LICENSE_ACKNOWLEDGED:
    raise RuntimeError("Read the DINOv3 licence/access instructions above. This pipeline is not permissive-only.")
if not math.isfinite(HEIGHT_CM) or HEIGHT_CM <= 0:
    raise ValueError("HEIGHT_CM must be positive and finite.")
if PIPELINE_TYPE not in ("512", "1024_cascade"):
    raise ValueError("Use 512 or 1024_cascade.")
if not isinstance(SEED, int) or not isinstance(STEPS, int) or STEPS < 1:
    raise ValueError("Use integer SEED and positive integer STEPS.")
if not isinstance(TARGET_FACES, int) or TARGET_FACES < 0 or not isinstance(MAX_NUM_TOKENS, int) or MAX_NUM_TOKENS < 1:
    raise ValueError("Invalid face/token limits.")
allowed_axes = {"auto", "+X", "-X", "+Y", "-Y", "+Z", "-Z"}
if SOURCE_UP not in allowed_axes or SOURCE_FORWARD not in allowed_axes:
    raise ValueError("Invalid source-axis override.")
install_software()
os.environ["HF_HOME"] = str(BUILD_ROOT / "hf-cache")
os.environ["U2NET_HOME"] = str(BUILD_ROOT / "rembg-cache")
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("Hugging Face read token (approved DINOv3 account): ")
# HEAD checks gated weights without uploading any images or logging the token.
run_command([ENV_PYTHON, "-c",
    "from huggingface_hub import get_hf_file_metadata, hf_hub_url; "
    "get_hf_file_metadata(hf_hub_url('facebook/dinov3-vitl16-pretrain-lvd1689m', 'model.safetensors', revision='" + DINO_REVISION + "')); "
    "print('Encoder access verified.')"])


## Upload, reconstruct, download, delete

Select all views in the file picker. Optional views are used for camera fits
only, never silently treated as multi-view shape conditioning. The original
pixel grid is preserved in cutouts/masks and camera `imageSize`, `originPx`,
`pxPerMeter`; yaw front/left/back/right is 0/90/180/270 degrees.
Enable browser downloads. The cell waits for the download transfer to browser
memory before removing its VM source; it cannot verify that you saved it to disk.


In [ ]:
EMBEDDED_FILES = {'shape_worker.py': '"""Isolated Colab worker; embedded into the notebook by build_notebook.py.\n\nUses the pinned TRELLIS.2 shape substeps, without BRIA or NVIDIA renderers.\nUser data and all derived files are confined to the session directory.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport os\nimport sys\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport trimesh\nfrom huggingface_hub import hf_hub_download, snapshot_download\nfrom PIL import Image\nfrom rembg import new_session, remove\n\nimport meshops\nfrom check_meta import validate_pair\n\nMODEL_REPO = "microsoft/TRELLIS.2-4B"\nMODEL_REVISION = "af44b45f2e35a493886929c6d786e563ec68364d"\n\n\ndef load_pipeline(config: dict):\n    from trellis2 import models\n    from trellis2.modules.image_feature_extractor import DinoV3FeatureExtractor\n    from trellis2.pipelines import Trellis2ImageTo3DPipeline, samplers\n\n    path = hf_hub_download(MODEL_REPO, "pipeline.json", revision=MODEL_REVISION)\n    arguments = json.loads(Path(path).read_text())["args"]\n    names = ["sparse_structure_flow_model", "sparse_structure_decoder", "shape_slat_decoder", "shape_slat_flow_model_512"]\n    if config["pipeline_type"] != "512":\n        names.append("shape_slat_flow_model_1024")\n    loaded = {}\n    for name in names:\n        checkpoint = arguments["models"][name]\n        if checkpoint.startswith("microsoft/"):\n            # The upstream sparse decoder belongs to the original MIT TRELLIS model.\n            repo, model = "/".join(checkpoint.split("/")[:2]), "/".join(checkpoint.split("/")[2:])\n            revision = config["sparse_decoder_revision"]\n        else:\n            repo, model, revision = MODEL_REPO, checkpoint, MODEL_REVISION\n        local = None\n        for extension in (".json", ".safetensors"):\n            local = hf_hub_download(repo, model + extension, revision=revision)\n        loaded[name] = models.from_pretrained(str(Path(local).with_suffix("")))\n    sparse_spec = arguments["sparse_structure_sampler"]\n    shape_spec = arguments["shape_slat_sampler"]\n    # Construct directly: upstream from_pretrained also creates BRIA RMBG-2.0.\n    pipeline = Trellis2ImageTo3DPipeline(\n        models=loaded,\n        sparse_structure_sampler=getattr(samplers, sparse_spec["name"])(**sparse_spec["args"]),\n        shape_slat_sampler=getattr(samplers, shape_spec["name"])(**shape_spec["args"]),\n        sparse_structure_sampler_params=sparse_spec["params"],\n        shape_slat_sampler_params=shape_spec["params"],\n        shape_slat_normalization=arguments["shape_slat_normalization"],\n        image_cond_model=DinoV3FeatureExtractor(model_name=snapshot_download(\n            arguments["image_cond_model"]["args"]["model_name"], revision=config["dino_revision"],\n            allow_patterns=["config.json", "*.safetensors", "*.safetensors.index.json"],\n        )),\n        rembg_model=None,\n        low_vram=True,\n        default_pipeline_type=config["pipeline_type"],\n    )\n    pipeline.cuda()\n    return pipeline\n\n\ndef prepare_cutouts(session_dir: Path) -> tuple[dict, dict, dict]:\n    session = new_session("u2net_human_seg", providers=["CPUExecutionProvider"])\n    cutouts, masks, inputs = {}, {}, {}\n    for view in meshops.VIEW_YAW_DEG:\n        path = session_dir / "uploads" / f"{view}.png"\n        if not path.exists():\n            continue\n        with Image.open(path) as source:\n            source.load()\n            cutout = remove(source.convert("RGB"), session=session).convert("RGBA")\n        alpha = np.asarray(cutout)[:, :, 3]\n        mask = (alpha > 127).astype(np.uint8) * 255\n        if not mask.any() or not (alpha > 204).any():\n            raise ValueError(f"No foreground found in {view}; use a clear full-body image.")\n        out = session_dir / "out"\n        cutout.save(out / f"cutout_{view}.png")\n        Image.fromarray(mask).save(out / f"mask_{view}.png")\n        cutouts[view], masks[view] = cutout, mask\n        inputs[view] = {\n            "file": f"{view}.png",\n            "backgroundRemoval": {"library": "rembg", "model": "u2net_human_seg", "threshold": 127},\n            "cutout": f"cutout_{view}.png",\n            "mask": f"mask_{view}.png",\n            "modelInput": None,\n        }\n    return cutouts, masks, inputs\n\n\ndef infer_shape(pipeline, image: Image.Image, config: dict):\n    # The RGBA image already has alpha; preprocess_image skips its rembg_model.\n    prepared = pipeline.preprocess_image(image)\n    torch.manual_seed(config["seed"])\n    parameters = {"steps": config["steps"]}\n    with torch.inference_mode():\n        condition_512 = pipeline.get_cond([prepared], 512)\n        coords = pipeline.sample_sparse_structure(condition_512, 32, 1, parameters)\n        if config["pipeline_type"] == "512":\n            latent = pipeline.sample_shape_slat(condition_512, pipeline.models["shape_slat_flow_model_512"], coords, parameters)\n            resolution = 512\n        else:\n            condition_1024 = pipeline.get_cond([prepared], 1024)\n            latent, resolution = pipeline.sample_shape_slat_cascade(\n                condition_512, condition_1024,\n                pipeline.models["shape_slat_flow_model_512"], pipeline.models["shape_slat_flow_model_1024"],\n                512, 1024, coords, parameters, max_num_tokens=config["max_num_tokens"],\n            )\n        generated, _ = pipeline.decode_shape_slat(latent, resolution)\n    return generated[0], prepared.size, resolution\n\n\ndef detection_for(vertices: np.ndarray, config: dict) -> dict:\n    detection = meshops.detect_frame(vertices)\n    for label, parameter in (("up", "source_up"), ("forward", "source_forward")):\n        axis = config[parameter]\n        if axis != "auto":\n            detection[f"{label}Axis"] = "XYZ".index(axis[1])\n            detection[f"{label}Sign"] = 1 if axis[0] == "+" else -1\n    if detection["upAxis"] == detection["forwardAxis"]:\n        raise ValueError("Source up and forward must be different axes; set both overrides.")\n    if config["flip_forward"]:\n        detection["forwardSign"] *= -1\n    detection["overrides"] = {key: config[key] for key in ("source_up", "source_forward", "flip_forward")}\n    detection["note"] = "Heuristic orientation; anatomical front must be checked locally. Re-run with explicit axis overrides if needed."\n    return detection\n\n\ndef main(session_dir: Path) -> None:\n    config = json.loads((session_dir / "config.json").read_text())\n    sys.path.insert(0, config["trellis_source"])\n    pipeline = load_pipeline(config)\n    cutouts, masks, inputs = prepare_cutouts(session_dir)\n    generated, crop_size, resolution = infer_shape(pipeline, cutouts["front"], config)\n    inputs["front"]["modelInput"] = {\n        "preprocessing": "upstream alpha > 0.8 bbox, square crop, black alpha composite, DINOv3 resize",\n        "cropSize": list(crop_size), "conditioningResolution": 512,\n    }\n    raw = trimesh.Trimesh(vertices=generated.vertices.cpu().numpy(), faces=generated.faces.cpu().numpy(), process=False)\n    mesh, hires, cleanup = meshops.cleanup(raw, target_faces=config["target_faces"])\n    detection = detection_for(np.asarray(mesh.vertices), config)\n    vertices, norm = meshops.normalize(np.asarray(mesh.vertices), detection, config["height_cm"] / 100)\n    mesh = trimesh.Trimesh(vertices=vertices, faces=mesh.faces, process=False)\n    trimesh.repair.fix_normals(mesh)\n    _ = mesh.vertex_normals\n    out = session_dir / "out"\n    mesh.export(out / "mesh.glb")\n    cameras = {\n        view: meshops.fit_orthographic_camera(np.asarray(mesh.vertices), np.asarray(mesh.faces), masks[view], yaw)\n        for view, yaw in meshops.VIEW_YAW_DEG.items() if view in masks\n    }\n    bounds = norm.final_bbox\n    meta = {\n        "schema": "twin-shape/1", "created": datetime.now(timezone.utc).isoformat(timespec="seconds"),\n        "name": "trellis2_colab",\n        "coordinateSystem": {\n            "units": "meters", "handedness": "right", "up": "+Y", "forward": "+Z (the character faces +Z)",\n            "characterLeft": "+X",\n            "origin": "x,z = middle of the bbox of the lowest 3 % of the mesh (feet contact patch); y = 0 at the lowest vertex (soles)",\n        },\n        "inputs": inputs,\n        "model": {\n            "repo": MODEL_REPO, "revision": MODEL_REVISION, "sourceRevision": config["trellis_revision"],\n            "multiView": False, "viewsUsedForShape": ["front"], "pipelineType": config["pipeline_type"],\n            "resolutionUsed": resolution, "seed": config["seed"], "steps": config["steps"], "offload": "low_vram",\n        },\n        "normalization": {\n            "sourceFrameDetection": detection, "rotationSourceToStandard": norm.rotation.tolist(),\n            "scaleMetersPerSourceUnit": norm.scale, "translationMeters": norm.translation.tolist(),\n            "matrix4x4RowMajor": norm.matrix().tolist(), "formula": "p_final = (R @ p_source) * scale + translation",\n            "heightCm": config["height_cm"], "heightNote": "total mesh height, including soles and hair",\n            "sourceBBox": norm.source_bbox,\n            "finalBBox": {**bounds, "sizeMeters": (np.array(bounds["max"]) - bounds["min"]).tolist()},\n        },\n        "cameras": {\n            "assumption": "orthographic; x\' = x*cos(yaw) - z*sin(yaw); u = originPx[0] + pxPerMeter*x\'; v = originPx[1] - pxPerMeter*y. Final world metres to ORIGINAL input pixels, v downwards; yaw front/left/back/right = 0/90/180/270. Fitted by silhouette IoU; perspective photos require refinement.",\n            "views": cameras,\n        },\n        "mesh": {"faces": len(mesh.faces), "vertices": len(mesh.vertices), "watertight": bool(mesh.is_watertight), "hiresFaces": len(hires.faces), "cleanup": cleanup},\n        "files": {"glb": "mesh.glb", "obj": None, "hiresGlb": None, "previews": None},\n        "licenseNote": "TRELLIS.2 code and 4B weights: MIT; required DINOv3 encoder: Meta DINOv3 custom license. Not a permissive-only pipeline. BRIA RMBG-2.0 and NVIDIA renderers are not used.",\n        "provenance": {"shape": "Microsoft TRELLIS.2-4B", "license": "MIT + DINOv3 custom license", "createdAt": datetime.now(timezone.utc).isoformat()},\n        "versions": {"torch": torch.__version__, "python": sys.version.split()[0]},\n    }\n    (out / "meta.json").write_text(json.dumps(meta, indent=2, allow_nan=False), encoding="utf-8")\n    validate_pair(out / "mesh.glb", out / "meta.json")\n    print("Shape exported and numeric convention checks passed. Check anatomical front locally.")\n\n\nif __name__ == "__main__":\n    main(Path(sys.argv[1]))\n', 'check_meta.py': '"""Check a shape-stage GLB and twin-shape/1 metadata without displaying private data.\n\nRequires numpy and trimesh (already used by the local shape stage).\nGeometry alone cannot prove which anatomical side is the front; inspect that locally.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport struct\nfrom pathlib import Path\n\nimport numpy as np\nimport trimesh\n\nVIEW_YAW_DEG = {"front": 0.0, "left": 90.0, "back": 180.0, "right": 270.0}\n\n\ndef _require(condition: bool, message: str) -> None:\n    if not condition:\n        raise ValueError(message)\n\n\ndef _array(value: object, shape: tuple[int, ...], label: str) -> np.ndarray:\n    result = np.asarray(value, dtype=float)\n    _require(result.shape == shape and bool(np.isfinite(result).all()), f"Invalid {label}")\n    return result\n\n\ndef project_points(points: np.ndarray, camera: dict) -> np.ndarray:\n    """Project final-frame metres to ORIGINAL-image pixels, with v downwards."""\n    yaw = np.deg2rad(camera["yawDeg"])\n    horizontal = points[:, 0] * np.cos(yaw) - points[:, 2] * np.sin(yaw)\n    scale = camera["pxPerMeter"]\n    origin = camera["originPx"]\n    return np.column_stack((origin[0] + scale * horizontal, origin[1] - scale * points[:, 1]))\n\n\ndef _embedded_glb(path: Path) -> None:\n    data = path.read_bytes()\n    _require(len(data) >= 20, "Truncated GLB")\n    magic, version, length = struct.unpack_from("<4sII", data)\n    _require(magic == b"glTF" and version == 2 and length == len(data), "Invalid GLB header")\n    size, kind = struct.unpack_from("<II", data, 12)\n    _require(kind == 0x4E4F534A and 20 + size <= len(data), "Missing GLB JSON chunk")\n    document = json.loads(data[20 : 20 + size])\n    for item in document.get("buffers", []) + document.get("images", []):\n        _require("uri" not in item or item["uri"].startswith("data:"), "External GLB resource")\n\n\ndef validate_pair(mesh_path: Path, meta_path: Path, tolerance_m: float = 2e-5) -> dict:\n    """Raise ValueError for an inconsistent bundle; return only a small numeric summary."""\n    _require(np.isfinite(tolerance_m) and tolerance_m > 0, "Invalid tolerance")\n    meta = json.loads(meta_path.read_text(encoding="utf-8"))\n    _require(meta.get("schema") == "twin-shape/1", "Expected twin-shape/1")\n    system = meta["coordinateSystem"]\n    for key, expected in {"units": "meters", "handedness": "right", "up": "+Y", "characterLeft": "+X"}.items():\n        _require(system.get(key) == expected, f"Invalid coordinateSystem.{key}")\n    _require(system.get("forward") == "+Z (the character faces +Z)", "Invalid forward convention")\n    _require(meta["files"]["glb"] == "mesh.glb", "Expected mesh.glb output")\n    _embedded_glb(mesh_path)\n    scene = trimesh.load_scene(mesh_path, process=False)\n    mesh = scene.to_geometry()\n    _require(isinstance(mesh, trimesh.Trimesh), "GLB must contain triangle geometry")\n    vertices = np.asarray(mesh.vertices)\n    _require(len(vertices) > 0 and len(mesh.faces) > 0 and bool(np.isfinite(vertices).all()), "Empty or nonfinite mesh")\n    bounds = np.array([vertices.min(0), vertices.max(0)])\n    normalization = meta["normalization"]\n    height = float(normalization["heightCm"]) / 100\n    _require(np.isfinite(height) and height > 0, "Invalid heightCm")\n    _require(abs(bounds[0, 1]) <= tolerance_m, "Feet must be at y=0")\n    _require(abs(bounds[1, 1] - bounds[0, 1] - height) <= tolerance_m, "Mesh height disagrees with heightCm")\n    feet = vertices[vertices[:, 1] < bounds[0, 1] + 0.03 * height]\n    center = (feet.min(0) + feet.max(0)) / 2\n    _require(bool(np.abs(center[[0, 2]]).max() <= tolerance_m), "Feet contact patch must be centered in x,z")\n    final = normalization["finalBBox"]\n    for key, actual in (("min", bounds[0]), ("max", bounds[1]), ("sizeMeters", bounds[1] - bounds[0])):\n        claimed = _array(final[key], (3,), f"finalBBox.{key}")\n        _require(bool(np.allclose(claimed, actual, atol=tolerance_m, rtol=0)), f"Incorrect finalBBox.{key}")\n    rotation = _array(normalization["rotationSourceToStandard"], (3, 3), "rotation")\n    _require(bool(np.allclose(rotation @ rotation.T, np.eye(3), atol=1e-6, rtol=0)), "Rotation must be orthonormal")\n    _require(abs(np.linalg.det(rotation) - 1) < 1e-6, "Rotation must be proper (no reflection)")\n    scale = float(normalization["scaleMetersPerSourceUnit"])\n    _require(np.isfinite(scale) and scale > 0, "Scale must be finite and positive")\n    translation = _array(normalization["translationMeters"], (3,), "translation")\n    transform = _array(normalization["matrix4x4RowMajor"], (4, 4), "matrix")\n    expected = np.eye(4)\n    expected[:3, :3] = rotation * scale\n    expected[:3, 3] = translation\n    _require(bool(np.allclose(transform, expected, atol=1e-7, rtol=0)), "Matrix disagrees with R, scale, translation")\n    for key, actual in (("vertices", len(vertices)), ("faces", len(mesh.faces))):\n        _require(meta["mesh"][key] == actual, f"Incorrect mesh.{key}")\n    _require(meta["mesh"]["watertight"] == bool(mesh.is_watertight), "Incorrect mesh.watertight")\n    inputs = meta["inputs"]\n    cameras = meta["cameras"]["views"]\n    _require("front" in inputs and set(inputs) == set(cameras), "One camera is required per input, including front")\n    for view, camera in cameras.items():\n        _require(view in VIEW_YAW_DEG, "Unknown camera view")\n        _require(camera.get("yawDeg") == VIEW_YAW_DEG[view], f"Incorrect {view} yaw")\n        image_size = _array(camera["imageSize"], (2,), "imageSize")\n        _require(bool((image_size > 0).all() and (image_size == np.floor(image_size)).all()), "Invalid imageSize")\n        _array(camera["originPx"], (2,), "originPx")\n        px_per_meter = float(camera["pxPerMeter"])\n        _require(np.isfinite(px_per_meter) and px_per_meter > 0, "Invalid pxPerMeter")\n        iou = float(camera["silhouetteIoU"])\n        _require(np.isfinite(iou) and 0 <= iou <= 1, "Invalid silhouetteIoU")\n        bbox = _array(camera["maskBBoxPx"], (4,), "maskBBoxPx")\n        _require(bool((bbox >= 0).all() and (bbox[:2] <= bbox[2:]).all() and (bbox[2:] < image_size).all()), "Invalid maskBBoxPx")\n    model = meta["model"]\n    used = model["viewsUsedForShape"]\n    _require(bool(used) and set(used).issubset(inputs), "Invalid viewsUsedForShape")\n    if model.get("repo") == "microsoft/TRELLIS.2-4B":\n        _require(model["multiView"] is False and used == ["front"], "TRELLIS.2 does not support multi-view conditioning")\n    return {"valid": True, "heightCm": height * 100, "cameraViews": list(cameras), "anatomicalFrontVerified": False}\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("mesh", type=Path)\n    parser.add_argument("meta", type=Path)\n    args = parser.parse_args()\n    try:\n        result = validate_pair(args.mesh, args.meta)\n    except (ValueError, KeyError, TypeError, OSError) as error:\n        parser.exit(1, f"Validation failed: {error}\\n")\n    print(json.dumps(result))\n\n\nif __name__ == "__main__":\n    main()\n', 'meshops.py': '"""Mesh post-processing, coordinate normalisation and silhouette camera fitting for the shape lab.\n\nEverything here is plain numpy / trimesh / scipy / OpenCV (no torch), so it can be unit-tested and reused by the\ntexture agent. Conventions of the OUTPUT (same as the web app, see docs/ARCHITECTURE.md): metres, right handed,\n+Y up, the character faces +Z (so the character\'s left is +X), feet on y = 0.\n"""\n\nfrom __future__ import annotations\n\nimport math\nfrom dataclasses import dataclass, field\n\nimport cv2\nimport numpy as np\nimport trimesh\nfrom scipy.optimize import minimize\n\n# --------------------------------------------------------------------------------------------------------------------\n# cleanup\n# --------------------------------------------------------------------------------------------------------------------\n\n\ndef _rebuild(vertices: np.ndarray, faces: np.ndarray) -> trimesh.Trimesh:\n    mesh = trimesh.Trimesh(vertices=np.asarray(vertices, dtype=np.float64), faces=np.asarray(faces), process=False)\n    mesh.merge_vertices()\n    mesh.update_faces(mesh.nondegenerate_faces())\n    mesh.update_faces(mesh.unique_faces())\n    mesh.remove_unreferenced_vertices()\n    return mesh\n\n\ndef keep_main_components(mesh: trimesh.Trimesh, floater_ratio: float = 0.02) -> tuple[trimesh.Trimesh, dict]:\n    """Keep the largest connected component (by surface area) plus any component whose area is at least\n    `floater_ratio` of the largest one. Everything else is a floater and is dropped."""\n    parts = mesh.split(only_watertight=False)\n    if len(parts) <= 1:\n        return mesh, {"components": len(parts), "kept": len(parts), "removedFaces": 0}\n    areas = np.array([p.area for p in parts])\n    main = float(areas.max())\n    keep = [p for p, a in zip(parts, areas, strict=True) if a >= floater_ratio * main]\n    removed_faces = int(sum(len(p.faces) for p in parts) - sum(len(p.faces) for p in keep))\n    merged = trimesh.util.concatenate(keep) if len(keep) > 1 else keep[0]\n    return merged, {"components": len(parts), "kept": len(keep), "removedFaces": removed_faces}\n\n\ndef decimate(mesh: trimesh.Trimesh, target_faces: int, agg: float = 6.0) -> trimesh.Trimesh:\n    """Quadric edge-collapse decimation to about `target_faces` triangles (fast_simplification)."""\n    if target_faces <= 0 or len(mesh.faces) <= target_faces:\n        return mesh\n    import fast_simplification\n\n    pts, tris = fast_simplification.simplify(\n        np.ascontiguousarray(mesh.vertices, dtype=np.float64),\n        np.ascontiguousarray(mesh.faces, dtype=np.int32),\n        target_count=int(target_faces),\n        agg=agg,\n    )\n    return _rebuild(pts, tris)\n\n\ndef cleanup(\n    mesh: trimesh.Trimesh,\n    floater_ratio: float = 0.02,\n    smooth_iters: int = 0,\n    target_faces: int = 0,\n) -> tuple[trimesh.Trimesh, trimesh.Trimesh, dict]:\n    """Full cleanup. Returns (final mesh, high-res cleaned mesh, stats).\n\n    order: weld/degenerate removal -> floaters -> hole fill -> (optional Taubin smoothing) -> normals -> decimation ->\n    normals again. The high-res mesh is the cleaned mesh BEFORE decimation."""\n    stats: dict = {"inputVertices": int(len(mesh.vertices)), "inputFaces": int(len(mesh.faces))}\n    mesh = _rebuild(mesh.vertices, mesh.faces)\n    mesh, comp = keep_main_components(mesh, floater_ratio)\n    stats["components"] = comp\n    mesh = _rebuild(mesh.vertices, mesh.faces)\n    if not mesh.is_watertight:\n        trimesh.repair.fill_holes(mesh)\n    if smooth_iters > 0:\n        trimesh.smoothing.filter_taubin(mesh, iterations=smooth_iters)\n    trimesh.repair.fix_normals(mesh)\n    stats["hiresFaces"] = int(len(mesh.faces))\n    stats["hiresWatertight"] = bool(mesh.is_watertight)\n    hires = mesh.copy()\n    if target_faces > 0:\n        mesh = decimate(mesh, target_faces)\n        trimesh.repair.fix_normals(mesh)\n    stats["finalVertices"] = int(len(mesh.vertices))\n    stats["finalFaces"] = int(len(mesh.faces))\n    stats["finalWatertight"] = bool(mesh.is_watertight)\n    return mesh, hires, stats\n\n\n# --------------------------------------------------------------------------------------------------------------------\n# orientation and normalisation\n# --------------------------------------------------------------------------------------------------------------------\n\n\ndef _slab_bbox_area(v: np.ndarray, axis: int, lo: float, hi: float) -> float:\n    sel = v[(v[:, axis] >= lo) & (v[:, axis] <= hi)]\n    if len(sel) < 3:\n        return 0.0\n    lat = [a for a in range(3) if a != axis]\n    ext = sel[:, lat].max(0) - sel[:, lat].min(0)\n    return float(ext[0] * ext[1])\n\n\ndef detect_frame(v: np.ndarray, default_up: int = 1, default_forward: int = 2) -> dict:\n    """Detect which source axis is \'up\' and which is \'forward\' for a standing person.\n\n    up axis      : the axis with the largest extent; its sign from the footprint area of the lowest 3 % slab (two\n                   shoes, large) versus the highest 3 % slab (top of the head, small).\n    forward axis : the horizontal axis with the smaller extent (chest depth < shoulder / arm span); its sign from the\n                   fact that the soles\' centroid lies in front of the shank centroid (toes point forward).\n    Returns the detection with a confidence flag and the fallback used when a cue is ambiguous."""\n    ext = v.max(0) - v.min(0)\n    up = int(np.argmax(ext))\n    lo, height = float(v[:, up].min()), float(ext[up])\n    bottom = _slab_bbox_area(v, up, lo, lo + 0.03 * height)\n    top = _slab_bbox_area(v, up, lo + 0.97 * height, lo + height)\n    ratio = bottom / top if top > 0 else float("inf")\n    if ratio > 1.5:\n        up_sign, up_conf = +1, True\n    elif ratio < 1 / 1.5:\n        up_sign, up_conf = -1, True\n    else:\n        up_sign, up_conf = +1, False  # ambiguous: assume the generator\'s native +Y up\n\n    lat = [a for a in range(3) if a != up]\n    forward = lat[0] if ext[lat[0]] < ext[lat[1]] else lat[1]\n    fwd_conf = abs(ext[lat[0]] - ext[lat[1]]) > 0.03 * height\n    if not fwd_conf:\n        forward = default_forward if default_forward in lat else lat[0]\n\n    t = (v[:, up] - lo) / height\n    if up_sign < 0:\n        t = 1.0 - t\n    foot = v[t < 0.04]\n    shank = v[(t > 0.12) & (t < 0.28)]\n    diff = float(foot[:, forward].mean() - shank[:, forward].mean()) if len(foot) and len(shank) else 0.0\n    fwd_sign = +1 if diff >= 0 else -1\n    sign_conf = abs(diff) > 0.01 * height\n    if not sign_conf:\n        fwd_sign = +1\n    return {\n        "upAxis": up,\n        "upSign": up_sign,\n        "forwardAxis": forward,\n        "forwardSign": fwd_sign,\n        "confident": bool(up_conf and fwd_conf and sign_conf),\n        "cues": {\n            "footprintBottomOverTop": None if math.isinf(ratio) else round(ratio, 3),\n            "extentsSource": [round(float(e), 4) for e in ext],\n            "soleMinusShankAlongForward": round(diff, 4),\n        },\n    }\n\n\ndef rotation_from_detection(det: dict) -> np.ndarray:\n    """3x3 rotation R with  v_std = R @ v_src  so that source up -> +Y and source forward -> +Z (proper rotation)."""\n    up_vec = np.zeros(3)\n    up_vec[det["upAxis"]] = det["upSign"]\n    fwd_vec = np.zeros(3)\n    fwd_vec[det["forwardAxis"]] = det["forwardSign"]\n    x_vec = np.cross(up_vec, fwd_vec)  # y cross z = x in a right handed frame\n    # rows of R are the images of the standard axes expressed in the source frame\n    return np.stack([x_vec, up_vec, fwd_vec], axis=0)\n\n\n@dataclass\nclass Normalization:\n    rotation: np.ndarray  # 3x3\n    scale: float  # metres per source unit\n    translation: np.ndarray  # 3, applied after rotation and scale\n    height_m: float\n    source_bbox: dict = field(default_factory=dict)\n    final_bbox: dict = field(default_factory=dict)\n\n    def matrix(self) -> np.ndarray:\n        m = np.eye(4)\n        m[:3, :3] = self.rotation * self.scale\n        m[:3, 3] = self.translation\n        return m\n\n\ndef normalize(v: np.ndarray, det: dict, height_m: float) -> tuple[np.ndarray, Normalization]:\n    """Rotate to the standard frame, scale the height to `height_m`, put the feet on y = 0 and centre x / z on the\n    contact patch of the feet (mid of the bbox of the lowest 3 % slab)."""\n    src_min, src_max = v.min(0), v.max(0)\n    r = rotation_from_detection(det)\n    vr = v @ r.T\n    h_src = float(vr[:, 1].max() - vr[:, 1].min())\n    scale = height_m / h_src\n    vs = vr * scale\n    y_min = float(vs[:, 1].min())\n    foot = vs[vs[:, 1] < y_min + 0.03 * height_m]\n    cx = 0.5 * float(foot[:, 0].min() + foot[:, 0].max())\n    cz = 0.5 * float(foot[:, 2].min() + foot[:, 2].max())\n    t = np.array([-cx, -y_min, -cz])\n    out = vs + t\n    norm = Normalization(\n        rotation=r,\n        scale=scale,\n        translation=t,\n        height_m=height_m,\n        source_bbox={"min": src_min.tolist(), "max": src_max.tolist()},\n        final_bbox={"min": out.min(0).tolist(), "max": out.max(0).tolist()},\n    )\n    return out, norm\n\n\n# --------------------------------------------------------------------------------------------------------------------\n# silhouette camera fit\n# --------------------------------------------------------------------------------------------------------------------\n\nVIEW_YAW_DEG = {"front": 0.0, "left": 90.0, "back": 180.0, "right": 270.0}\n\n\ndef project_x(v: np.ndarray, yaw_deg: float) -> np.ndarray:\n    """Horizontal image coordinate (metres, +right) of an orthographic camera on the +Z side rotated by `yaw_deg`\n    about +Y (0 front, 90 = camera on the character\'s left side (+X), 180 back, 270 right side)."""\n    a = math.radians(yaw_deg)\n    return v[:, 0] * math.cos(a) - v[:, 2] * math.sin(a)\n\n\ndef raster_silhouette(\n    v: np.ndarray,\n    f: np.ndarray,\n    yaw_deg: float,\n    s: float,\n    u0: float,\n    v0: float,\n    w: int,\n    h: int,\n    ds: float = 0.5,\n    mirror: bool = False,\n    sy: float | None = None,\n) -> np.ndarray:\n    """Binary silhouette (uint8 0/255) of the mesh under  u = u0 + s * x\',  v = v0 - s * y  at scale `ds`.\n    `mirror` flips x\' (mirror about the image column of the world origin), used as a handedness sanity check.\n    `sy` optional separate vertical scale (px per metre) for the anisotropic diagnostic fit."""\n    xp = project_x(v, yaw_deg)\n    if mirror:\n        xp = -xp\n    pts = np.stack([(u0 + s * xp) * ds, (v0 - (s if sy is None else sy) * v[:, 1]) * ds], axis=1)\n    tri = np.round(pts[f] * 16.0).astype(np.int32)\n    img = np.zeros((max(1, int(round(h * ds))), max(1, int(round(w * ds)))), dtype=np.uint8)\n    # one convex fill per triangle: cv2.fillPoly over many contours uses the even-odd rule, which would punch holes\n    # wherever front and back surfaces overlap in the projection.\n    for t in tri:\n        cv2.fillConvexPoly(img, t, 255, lineType=cv2.LINE_8, shift=4)\n    return img\n\n\ndef _iou(a: np.ndarray, b: np.ndarray) -> float:\n    inter = np.logical_and(a, b).sum()\n    union = np.logical_or(a, b).sum()\n    return float(inter) / float(union) if union else 0.0\n\n\ndef alpha_bbox(mask: np.ndarray) -> tuple[int, int, int, int]:\n    ys, xs = np.nonzero(mask)\n    return int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())\n\n\ndef fit_orthographic_camera(\n    v: np.ndarray, f: np.ndarray, mask: np.ndarray, yaw_deg: float, ds: float = 0.5, max_faces: int = 25000\n) -> dict:\n    """Fit (px per metre, image position of the world origin) so the mesh silhouette matches the person mask.\n\n    Model:  u = u0 + s * x\'   (x\' = x cos yaw - z sin yaw),   v = v0 - s * y      (v grows downwards).\n    Initial guess: s from the mask bbox height vs. the mesh height, u0 from the bbox centre, v0 = bbox bottom (feet).\n    Then Nelder-Mead on (s, u0, v0) maximising the silhouette IoU."""\n    h_img, w_img = mask.shape[:2]\n    m = mask.astype(bool)  # any non-zero pixel is person (works for 0/1 and 0/255 masks)\n    x0, y0, x1, y1 = alpha_bbox(m)\n    height_m = float(v[:, 1].max() - v[:, 1].min())\n    s_init = (y1 - y0 + 1) / height_m\n    xp = project_x(v, yaw_deg)\n    u_init = 0.5 * (x0 + x1 + 1) - s_init * 0.5 * float(xp.min() + xp.max())\n    v_init = float(y1 + 1) + s_init * float(v[:, 1].min())\n\n    # proxy mesh for speed\n    if len(f) > max_faces:\n        try:\n            import fast_simplification\n\n            pv, pf = fast_simplification.simplify(\n                np.ascontiguousarray(v, dtype=np.float64), np.ascontiguousarray(f, dtype=np.int32), target_count=max_faces\n            )\n        except Exception:  # noqa: BLE001 - fall back to the full mesh\n            pv, pf = v, f\n    else:\n        pv, pf = v, f\n\n    target = cv2.resize(m.astype(np.uint8), (max(1, int(round(w_img * ds))), max(1, int(round(h_img * ds)))), interpolation=cv2.INTER_AREA) > 0\n\n    def loss(p: np.ndarray) -> float:\n        s, u0, v0 = p\n        sil = raster_silhouette(pv, pf, yaw_deg, s, u0, v0, w_img, h_img, ds) > 0\n        return 1.0 - _iou(sil, target)\n\n    iou_init = 1.0 - loss(np.array([s_init, u_init, v_init]))\n    simplex = np.array(\n        [\n            [s_init, u_init, v_init],\n            [s_init * 1.02, u_init, v_init],\n            [s_init, u_init + 8.0, v_init],\n            [s_init, u_init, v_init + 8.0],\n        ]\n    )\n    res = minimize(loss, x0=np.array([s_init, u_init, v_init]), method="Nelder-Mead",\n                   options={"initial_simplex": simplex, "xatol": 0.05, "fatol": 1e-5, "maxiter": 400})\n    s, u0, v0 = (float(x) for x in res.x)\n    iou_fit = 1.0 - float(res.fun)\n    if iou_fit < iou_init:  # never accept a worse fit\n        s, u0, v0, iou_fit = s_init, u_init, v_init, iou_init\n    # diagnostic: separate horizontal / vertical scale (how much the generated proportions deviate from the photo)\n    def loss_aniso(p: np.ndarray) -> float:\n        sx, sy_, u0_, v0_ = p\n        sil = raster_silhouette(pv, pf, yaw_deg, sx, u0_, v0_, w_img, h_img, ds, sy=sy_) > 0\n        return 1.0 - _iou(sil, target)\n\n    simplex4 = np.array(\n        [[s, s, u0, v0], [s * 1.02, s, u0, v0], [s, s * 1.02, u0, v0], [s, s, u0 + 8.0, v0], [s, s, u0, v0 + 8.0]]\n    )\n    res2 = minimize(loss_aniso, x0=np.array([s, s, u0, v0]), method="Nelder-Mead",\n                    options={"initial_simplex": simplex4, "xatol": 0.05, "fatol": 1e-5, "maxiter": 500})\n    aniso = {\n        "pxPerMeterX": float(res2.x[0]),\n        "pxPerMeterY": float(res2.x[1]),\n        "originPx": [float(res2.x[2]), float(res2.x[3])],\n        "silhouetteIoU": round(1.0 - float(res2.fun), 4),\n        "horizontalOverVertical": round(float(res2.x[0] / res2.x[1]), 4),\n        "note": "diagnostic only: the generated body is this much narrower (<1) or wider (>1) relative to its height than the photo",\n    }\n    # mirror check: is a horizontally mirrored silhouette a better match? (mesh handedness sanity check)\n    iou_mirror = _iou(raster_silhouette(pv, pf, yaw_deg, s, u0, v0, w_img, h_img, ds, mirror=True) > 0, target)\n    return {\n        "yawDeg": yaw_deg,\n        "imageSize": [int(w_img), int(h_img)],\n        "pxPerMeter": s,\n        "originPx": [u0, v0],\n        "silhouetteIoU": round(iou_fit, 4),\n        "silhouetteIoUInitialGuess": round(iou_init, 4),\n        "silhouetteIoUMirroredMesh": round(iou_mirror, 4),\n        "maskBBoxPx": [x0, y0, x1, y1],\n        "initialGuess": {"pxPerMeter": s_init, "originPx": [u_init, v_init]},\n        "anisotropicFit": aniso,\n    }\n\n\ndef overlay_image(v: np.ndarray, f: np.ndarray, cam: dict, mask: np.ndarray, scale: float = 0.5) -> np.ndarray:\n    """BGR debug image: green = photo person mask, red = projected mesh silhouette, yellow = both."""\n    w, h = cam["imageSize"]\n    sil = raster_silhouette(v, f, cam["yawDeg"], cam["pxPerMeter"], cam["originPx"][0], cam["originPx"][1], w, h, ds=scale) > 0\n    m = cv2.resize(mask.astype(np.uint8), (sil.shape[1], sil.shape[0]), interpolation=cv2.INTER_AREA) > 0\n    out = np.zeros((*sil.shape, 3), dtype=np.uint8)\n    out[..., 1] = m * 200\n    out[..., 2] = sil * 200\n    return out\n'}


In [ ]:
import json
import zipfile
from google.colab import files

def run_session():
    session_dir = Path(tempfile.mkdtemp(prefix="dt-shape-session-", dir="/content"))
    uploaded = {}
    try:
        uploads_dir = session_dir / "uploads"
        uploads_dir.mkdir()
        (session_dir / "out").mkdir()
        config = {
            "height_cm": HEIGHT_CM, "seed": SEED, "pipeline_type": PIPELINE_TYPE,
            "steps": STEPS, "target_faces": TARGET_FACES, "max_num_tokens": MAX_NUM_TOKENS,
            "source_up": SOURCE_UP, "source_forward": SOURCE_FORWARD, "flip_forward": FLIP_FORWARD,
            "trellis_source": str(TRELLIS_SOURCE), "trellis_revision": TRELLIS_REVISION,
            "sparse_decoder_revision": SPARSE_DECODER_REVISION, "dino_revision": DINO_REVISION,
        }
        (session_dir / "config.json").write_text(json.dumps(config), encoding="utf-8")
        for name, source in EMBEDDED_FILES.items():
            (session_dir / name).write_text(source, encoding="utf-8")
        print("Upload front.png and optional back.png / left.png / right.png together.")
        # target_dir confines files.upload's on-disk copies, even on rejection.
        uploaded = files.upload(target_dir=str(uploads_dir))
        names = {Path(name).name for name in uploaded}
        if "front.png" not in names or not names.issubset({"front.png", "left.png", "back.png", "right.png"}):
            raise ValueError("Upload exactly front.png plus optional left.png/back.png/right.png.")
        uploaded.clear()  # No photo bytes kept in notebook globals.
        run_command([ENV_PYTHON, session_dir / "shape_worker.py", session_dir])
        archive = session_dir / "trellis2_shape.zip"
        with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
            for path in sorted((session_dir / "out").iterdir()):
                bundle.write(path, path.name)
        download_and_wait(archive)
        print("Archive transferred to the browser. Save it locally before closing this tab.")
    except BaseException:
        print("Session stopped; private VM files will be removed. Re-run this cell to retry.")
        raise
    finally:
        uploaded.clear()
        cleanup_session(session_dir)
        os.environ.pop("HF_TOKEN", None)

run_session()


## Privacy cell (runs automatically at the end)

The processing cell already removes uploads and every derived output in a
`finally` block. This final cell verifies that no session directory remains
and drops the session token. Public software/model caches are not personal
data and remain until you delete the runtime. Disconnect and delete this
runtime after use. Never commit the downloaded archive or its contents.


In [ ]:
remaining_sessions = list(Path("/content").glob("dt-shape-session-*"))
if remaining_sessions:
    raise RuntimeError("A prior interrupted session remains. Disconnect and delete this runtime.")
os.environ.pop("HF_TOKEN", None)
gc.collect()
print("No private session files remain. Now disconnect and delete this runtime.")
